![Estimer le prix des logements à Bordeaux Métropole](https://raw.githubusercontent.com/yayefatou-lam/prix-logements-bordeaux/main/assets/banner.png)

# Prix des logements à Bordeaux Métropole · 1. Les données

**Yaye Fatou Lam** · projet personnel de data science · notebook 1 sur 3

| $\color{#B5562F}{\mathbf{187\,909}}$ | $\color{#B5562F}{\mathbf{57\,244}}$ | $\color{#B5562F}{\mathbf{28}}$ | $\color{#B5562F}{\mathbf{8}}$ |
|:---:|:---:|:---:|:---:|
| ventes brutes en Gironde, 2021-2025 | ventes retenues | communes de Bordeaux Métropole | règles de cohérence documentées |

> $\color{#2F76AE}{\textbf{Sommaire}}$
>
> 1. Charger les données
> 2. Le piège : une vente sur plusieurs lignes
> 3. Filtrer les ventes
> 4. Une ligne par vente
> 5. Règles de cohérence
> 6. Sauvegarde et contrôle croisé

Ce notebook transforme le fichier public des ventes immobilières (DVF) en une table propre : **une ligne par vente d'un seul logement**, avec son prix au m². Chaque décision est justifiée et chiffrée ; le détail figure dans [`NETTOYAGE.md`](../NETTOYAGE.md).

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# Racine du dépôt : les notebooks sont dans notebooks/
RACINE = Path.cwd().parent
sys.path.insert(0, str(RACINE))

from src.affichage import fr, tableau

## $\color{#B5562F}{\mathbf{1}}$ &nbsp; Charger les données

Les données sont les **Demandes de valeurs foncières géolocalisées**, publiées par Etalab sur [data.gouv.fr](https://www.data.gouv.fr/datasets/demandes-de-valeurs-foncieres-geolocalisees) : chaque vente enregistrée chez un notaire, avec son prix, sa surface et sa position. On lit directement les fichiers de la Gironde, de 2021 à 2025.

> $\color{#8A6420}{\textbf{Point de vigilance}}$
>
> Les conditions d'utilisation de DVF interdisent l'indexation des données par les moteurs de recherche.
> Le dépôt ne contient donc aucune donnée : elles sont téléchargées à chaque exécution, et aucune adresse n'est affichée.

In [2]:
ANNEES = [2021, 2022, 2023, 2024, 2025]
URL = "https://files.data.gouv.fr/geo-dvf/latest/csv/{annee}/departements/33.csv.gz"

# Codes lus comme du texte pour garder les zéros en tête
types_texte = {"code_commune": str, "code_postal": str, "id_parcelle": str,
               "code_departement": str, "ancien_code_commune": str}

morceaux = []
for annee in ANNEES:
    df = pd.read_csv(URL.format(annee=annee), dtype=types_texte, low_memory=False)
    print(f"{annee} : {len(df)} lignes")
    morceaux.append(df)

dvf = pd.concat(morceaux, ignore_index=True)
dvf["date_mutation"] = pd.to_datetime(dvf["date_mutation"])
print(dvf.shape)

2021 : 118188 lignes
2022 : 119809 lignes
2023 : 92716 lignes
2024 : 82927 lignes
2025 : 111712 lignes
(525352, 40)


In [3]:
print("Nature des mutations :")
print(dvf["nature_mutation"].value_counts(), "\n")
print("Type de local :")
print(dvf["type_local"].value_counts(dropna=False))

Nature des mutations :
nature_mutation
Vente                                 482710
Vente en l'état futur d'achèvement     34886
Echange                                 5691
Adjudication                             962
Vente terrain à bâtir                    949
Expropriation                            154
Name: count, dtype: int64 

Type de local :
type_local
NaN                                         215426
Dépendance                                  126487
Maison                                      101226
Appartement                                  67066
Local industriel. commercial ou assimilé     15147
Name: count, dtype: int64


## $\color{#B5562F}{\mathbf{2}}$ &nbsp; Le piège : une vente sur plusieurs lignes

Dans DVF, une ligne correspond à un local **et** une parcelle. Une même vente s'étale donc souvent sur plusieurs lignes, et le prix total est **répété** sur chacune.

In [4]:
# Nombre de lignes par vente
lignes_par_vente = dvf.groupby("id_mutation").size()
print("Nombre de ventes :", lignes_par_vente.size)
print(lignes_par_vente.value_counts().sort_index().head(10))

Nombre de ventes : 187909
1     67930
2     61787
3     27921
4     12375
5      4944
6      4031
7      1959
8      1560
9      1033
10      742
Name: count, dtype: int64


In [5]:
# Exemple d'une vente à 4 lignes (sans les colonnes d'adresse)
cols = ["id_mutation", "date_mutation", "nature_mutation", "valeur_fonciere",
        "type_local", "surface_reelle_bati", "nombre_pieces_principales", "surface_terrain"]
exemple = lignes_par_vente[lignes_par_vente == 4].index[0]
exemple_vente = (dvf.loc[dvf["id_mutation"] == exemple, cols]
                 .rename(columns={"id_mutation": "Vente", "date_mutation": "Date", "nature_mutation": "Nature",
                                  "valeur_fonciere": "Prix (€)", "type_local": "Type de local",
                                  "surface_reelle_bati": "Surface (m²)", "nombre_pieces_principales": "Pièces",
                                  "surface_terrain": "Terrain (m²)"}))
exemple_vente["Date"] = exemple_vente["Date"].dt.strftime("%d/%m/%Y")
tableau(exemple_vente.set_index("Vente"),
        legende="Le prix total de la vente est répété sur chacune des 4 lignes.")

,Date,Nature,Prix (€),Type de local,Surface (m²),Pièces,Terrain (m²)
Vente,,,,,,,
2021-502618,11/01/2021,Vente,390 000,Dépendance,–,0,270
2021-502618,11/01/2021,Vente,390 000,Appartement,89,3,270
2021-502618,11/01/2021,Vente,390 000,Appartement,23,2,270
2021-502618,11/01/2021,Vente,390 000,–,–,–,57


> $\color{#55623F}{\textbf{Constat}}$
>
> Une vente compte en moyenne près de 3 lignes, et un tiers seulement des ventes tient sur une ligne.
> Quand une vente contient plusieurs logements, le prix de chacun est impossible à retrouver : ces ventes devront être écartées.

## $\color{#B5562F}{\mathbf{3}}$ &nbsp; Filtrer les ventes

Trois filtres successifs, en comptant les ventes restantes à chaque étape :
- les **28 communes de Bordeaux Métropole** ;
- les ventes **classiques** (nature « Vente ») : on écarte les ventes sur plan (VEFA), les échanges, les adjudications et les expropriations, dont les prix suivent un autre marché ;
- les ventes d'**un seul appartement ou d'une seule maison**, sans local commercial. Une vente « logement + garage » reste, et le nombre de dépendances devient une variable.

In [6]:
from src.entrainer import COMMUNES_BM, LOCAL_COM

etapes = {"Ventes en Gironde, 2021-2025": dvf["id_mutation"].nunique()}

bm = dvf[dvf["code_commune"].isin(COMMUNES_BM)]
etapes["dont Bordeaux Métropole"] = bm["id_mutation"].nunique()

bm = bm[bm["nature_mutation"] == "Vente"]
etapes["dont nature « Vente »"] = bm["id_mutation"].nunique()

# Composition de chaque vente : nombre de locaux de chaque type
compo = (bm.dropna(subset=["type_local"])
           .groupby(["id_mutation", "type_local"]).size()
           .unstack(fill_value=0))
compo["logements"] = compo["Appartement"] + compo["Maison"]

# Ventes gardées : un seul logement et aucun local commercial
garder = compo[(compo["logements"] == 1) & (compo[LOCAL_COM] == 0)].index
etapes["dont un seul logement, sans local commercial"] = len(garder)

print("Communes trouvées :", bm["code_commune"].nunique(), "sur 28")
print("\nLogements par vente :")
print(compo["logements"].value_counts().sort_index().head(8))
tableau(pd.Series(etapes, name="Ventes restantes"))

Communes trouvées : 28 sur 28

Logements par vente :
logements
0     6948
1    59229
2     2780
3      472
4      308
5      132
6      114
7       46
Name: count, dtype: int64


,Ventes restantes
"Ventes en Gironde, 2021-2025",187 909
dont Bordeaux Métropole,88 033
dont nature « Vente »,75 896
"dont un seul logement, sans local commercial",58 514


## $\color{#B5562F}{\mathbf{4}}$ &nbsp; Une ligne par vente

In [7]:
ventes = bm[bm["id_mutation"].isin(garder)]

# La ligne du logement : une seule par vente, par construction
cols = ["id_mutation", "date_mutation", "valeur_fonciere", "code_commune", "nom_commune",
        "type_local", "surface_reelle_bati", "nombre_pieces_principales", "longitude", "latitude"]
table = (ventes[ventes["type_local"].isin(["Appartement", "Maison"])][cols]
         .set_index("id_mutation"))

# Nombre de dépendances (garage, cave, parking) vendues avec le logement
table["nb_dependances"] = compo.loc[table.index, "Dépendance"]

# Surface de terrain : chaque couple (parcelle, type de sol) compté une seule fois
terrain = (ventes.drop_duplicates(["id_mutation", "id_parcelle", "code_nature_culture", "surface_terrain"])
                 .groupby("id_mutation")["surface_terrain"].sum(min_count=1))
table["surface_terrain"] = terrain

# Pour un appartement, c'est la parcelle de tout l'immeuble : on ne la garde pas
table.loc[table["type_local"] == "Appartement", "surface_terrain"] = np.nan

# Variable cible
table["prix_m2"] = table["valeur_fonciere"] / table["surface_reelle_bati"]

n_types = table["type_local"].value_counts()
print(f"{fr(len(table))} ventes : {fr(n_types['Appartement'])} appartements et {fr(n_types['Maison'])} maisons")

# Résumé des variables, avec des noms et des nombres lisibles
resume = (
    table[["valeur_fonciere", "surface_reelle_bati", "nombre_pieces_principales",
           "surface_terrain", "prix_m2"]]
    .describe(percentiles=[.01, .5, .99])
    .rename(columns={"valeur_fonciere": "Prix (€)", "surface_reelle_bati": "Surface (m²)",
                     "nombre_pieces_principales": "Pièces", "surface_terrain": "Terrain (m²)",
                     "prix_m2": "Prix au m² (€)"},
            index={"count": "Nombre", "mean": "Moyenne", "std": "Écart-type", "min": "Minimum",
                   "1%": "1er centile", "50%": "Médiane", "99%": "99e centile", "max": "Maximum"})
)
tableau(resume,
        sauge=["1er centile", "Médiane", "99e centile"],
        ocre=["Minimum", "Maximum"],
        legende="En vert : la zone des biens courants · en ocre : les valeurs extrêmes, à nettoyer.")

58 514 ventes : 35 747 appartements et 22 767 maisons


,Prix (€),Surface (m²),Pièces,Terrain (m²),Prix au m² (€)
Nombre,58 486,58 513,58 513,20 721,58 485
Moyenne,304 685,74,3,524,4 151
Écart-type,212 206,39,1,1 224,1 849
Minimum,1,2,0,1,0
1er centile,69 000,18,1,50,1 309
Médiane,249 000,68,3,355,3 951
99e centile,1 079 405,204,7,2 706,8 811
Maximum,5 100 000,550,54,43 706,242 500


> $\color{#8A6420}{\textbf{Point de vigilance}}$
>
> Pour un appartement, `surface_terrain` est la surface de la parcelle de **tout l'immeuble**, pas un jardin : elle est vidée.
> Les valeurs extrêmes (prix de 1 €, surfaces de 2 m², prix au m² de plus de 200 000 €) montrent qu'un tri reste nécessaire.

## $\color{#B5562F}{\mathbf{5}}$ &nbsp; Règles de cohérence

Plutôt que de couper aux centiles (ce qui écarterait des ventes valides par construction), on fixe des **seuils justifiables**. La cellule ci-dessous compte d'abord les ventes visées par chaque règle.

In [8]:
regles = {
    "prix ou surface manquants": table["prix_m2"].isna(),
    "coordonnées manquantes": table["latitude"].isna() | table["longitude"].isna(),
    "surface < 9 m²": table["surface_reelle_bati"] < 9,              # Minimum légal d'un logement décent
    "prix au m² < 1 000 €": table["prix_m2"] < 1000,                 # Prix symboliques
    "prix au m² > 15 000 €": table["prix_m2"] > 15000,               # Surfaces mal saisies
    "0 pièce": table["nombre_pieces_principales"] == 0,
    "plus de 15 pièces": table["nombre_pieces_principales"] > 15,
    "plus de 5 dépendances": table["nb_dependances"] > 5,            # Logement vendu avec des parkings en lot
}

# Décompte par type de logement
impact = pd.DataFrame({nom: m.groupby(table["type_local"]).sum() for nom, m in regles.items()}).T
impact["total"] = impact.sum(axis=1)

# Une vente peut tomber sous plusieurs règles : total sans double compte
a_ecarter = pd.concat(regles, axis=1).any(axis=1)
print(f"Ventes écartées : {fr(a_ecarter.sum())} ({fr(100 * a_ecarter.mean(), 1)} %)")
tableau(impact)

Ventes écartées : 1 270 (2,2 %)


type_local,Appartement,Maison,total
prix ou surface manquants,18,11,29
coordonnées manquantes,464,313,777
surface < 9 m²,3,1,4
prix au m² < 1 000 €,152,213,365
prix au m² > 15 000 €,23,30,53
0 pièce,22,1,23
plus de 15 pièces,3,2,5
plus de 5 dépendances,22,4,26


In [9]:
# Les cas extrêmes, sans adresse
NOMS_VUE = {"date_mutation": "Date", "nom_commune": "Commune", "type_local": "Type",
            "valeur_fonciere": "Prix (€)", "surface_reelle_bati": "Surface (m²)",
            "nombre_pieces_principales": "Pièces", "prix_m2": "Prix au m² (€)"}


def vue(df):
    """Colonnes utiles, noms lisibles et dates au format français."""
    v = df[list(NOMS_VUE)].rename(columns=NOMS_VUE).rename_axis("Vente")
    v["Date"] = v["Date"].dt.strftime("%d/%m/%Y")
    return v


display(tableau(vue(table.sort_values("prix_m2").head(5)),
                legende="Les prix au m² les plus bas : des ventes à 1 €."))
display(tableau(vue(table.sort_values("prix_m2", ascending=False).head(5)),
                legende="Les prix au m² les plus hauts : des surfaces mal saisies."))

,Date,Commune,Type,Prix (€),Surface (m²),Pièces,Prix au m² (€)
Vente,,,,,,,
2021-519500,02/11/2021,Bordeaux,Appartement,1,210,4,0
2024-366511,29/01/2024,Bordeaux,Appartement,1,146,4,0
2022-496352,04/02/2022,Bordeaux,Appartement,1,124,6,0
2022-537073,30/11/2022,Bordeaux,Appartement,1,120,4,0
2024-366608,27/02/2024,Bordeaux,Appartement,1,114,5,0


,Date,Commune,Type,Prix (€),Surface (m²),Pièces,Prix au m² (€)
Vente,,,,,,,
2021-511124,14/06/2021,Saint-Médard-en-Jalles,Maison,485 000,2,6,242 500
2022-496243,31/01/2022,Bordeaux,Appartement,900 000,12,1,75 000
2023-419158,27/07/2023,Bordeaux,Maison,1 656 787,30,2,55 226
2021-508664,17/05/2021,Eysines,Appartement,379 000,7,4,54 143
2023-418538,18/12/2023,Bordeaux,Appartement,625 000,15,1,41 667


> $\color{#55623F}{\textbf{Constat}}$
>
> Les prix les plus bas sont des ventes à **1 €** : des transferts sans prix de marché.
> Les prix les plus hauts viennent d'une **surface mal saisie** (une maison de 6 pièces déclarée à 2 m²), pas d'un prix aberrant.
> Au total, environ 2 % des ventes sont écartées.

## $\color{#B5562F}{\mathbf{6}}$ &nbsp; Sauvegarde et contrôle croisé

In [10]:
propre = table[~a_ecarter].copy()
etapes["dont données cohérentes (8 règles)"] = len(propre)

# Table nettoyée pour les notebooks suivants (dossier data/, jamais publié)
(RACINE / "data").mkdir(exist_ok=True)
propre.to_parquet(RACINE / "data" / "ventes_bm_2021_2025.parquet")

n_types = propre["type_local"].value_counts()
print(f"Table finale : {fr(n_types['Appartement'])} appartements et {fr(n_types['Maison'])} maisons")
tableau(pd.Series(etapes, name="Ventes restantes"), sauge=["dont données cohérentes (8 règles)"])

Table finale : 35 046 appartements et 22 198 maisons


,Ventes restantes
"Ventes en Gironde, 2021-2025",187 909
dont Bordeaux Métropole,88 033
dont nature « Vente »,75 896
"dont un seul logement, sans local commercial",58 514
dont données cohérentes (8 règles),57 244


Le script de production [`src/entrainer.py`](../src/entrainer.py), qui réentraîne le modèle de l'application, refait ce nettoyage de son côté. On vérifie qu'il obtient **exactement** la même table.

In [11]:
from src.entrainer import nettoyer

table_script = nettoyer(dvf)
pd.testing.assert_frame_equal(propre.sort_index(), table_script.sort_index())
print("Contrôle croisé réussi : le notebook et le script produisent la même table.")

  Ventes en Gironde                                 187909
  dont Bordeaux Métropole                            88033
  dont nature « Vente »                              75896
  dont un seul logement, sans local commercial       58514
  dont données cohérentes (8 règles)                 57244
Contrôle croisé réussi : le notebook et le script produisent la même table.
